# 02 — Experiment A: Frozen representation classification

This notebook evaluates four frozen ViT-S/16 encoder conditions at three labelled-data fractions and three downstream seeds.

- Encoders: **Random initialization**, **ImageNet**, **MAE-style**, and **EMA-distillation**.
- Label fractions: **25%, 50%, 100%** of the fixed CUBS training split.
- Seeds: **42, 43, 44**.

Historical internal names containing `IJEPA_clean_2840` refer to the EMA-distillation condition and are retained to preserve compatibility with the outputs.

## 1. Environment and paths

In [ ]:
!pip -q install timm scikit-learn pandas tqdm pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, json, random, pickle, copy, gc
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import timm, torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(42)
print('Device:', DEVICE)
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
class Config:
    PROJECT_DIR = '/content/drive/MyDrive/IJEPA_Carotid_ViTS16'
    CLEAN_RUN = os.path.join(PROJECT_DIR, 'clean_2840_20260930')
    CHECKPOINT_DIR = os.path.join(CLEAN_RUN, 'checkpoints')
    RESULT_DIR = os.path.join(CLEAN_RUN, 'results')
    MANIFEST_DIR = os.path.join(CLEAN_RUN, 'manifests')
    STATE_DIR = os.path.join(CLEAN_RUN, 'resume_state')
    REVIEW_DIR = os.path.join(CLEAN_RUN, 'review')

    ARCH = 'vit_small_patch16_224'
    IMG_SIZE = 224
    DOWNSTREAM_SEEDS = [42, 43, 44]
    LABEL_FRACTIONS = [0.25, 0.50, 1.00]

    CLASS_BATCH_SIZE = 32
    CLASS_EPOCHS = 80
    CLASS_PATIENCE = 12
    CLASS_LR = 1e-3

for p in [Config.CHECKPOINT_DIR, Config.RESULT_DIR, Config.MANIFEST_DIR, Config.STATE_DIR, Config.REVIEW_DIR]:
    os.makedirs(p, exist_ok=True)

print('Clean namespace:', Config.CLEAN_RUN)

## 2. Load the CUBS split and SSL checkpoints

In [ ]:
CACHE = os.path.join(Config.MANIFEST_DIR, 'cubs_processed_split_clean.pkl')
IJEPA_CKPT = os.path.join(Config.CHECKPOINT_DIR, 'ijepa_corrected_2840_encoder.pth')
MAE_CKPT = os.path.join(Config.CHECKPOINT_DIR, 'mae_corrected_2840_encoder.pth')

for p in [CACHE, IJEPA_CKPT, MAE_CKPT]:
    assert os.path.exists(p), f'Missing required clean artifact: {p}'

with open(CACHE, 'rb') as f:
    d = pickle.load(f)

cubs_train_records = d['cubs_train_records']
cubs_val_records = d['cubs_val_records']
cubs_test_records = d['cubs_test_records']

assert (len(cubs_train_records), len(cubs_val_records), len(cubs_test_records)) == (1740, 218, 218)

print('CUBS split:', len(cubs_train_records), len(cubs_val_records), len(cubs_test_records))
print('Proxy threshold:', d.get('proxy_threshold', d.get('threshold', 'stored in cache')))

for name, path in [('I-JEPA', IJEPA_CKPT), ('MAE', MAE_CKPT)]:
    ck = torch.load(path, map_location='cpu')
    assert 'encoder_state_dict' in ck, f'{name}: encoder_state_dict missing'
    n = ck.get('n_ssl_images', 2840)
    assert int(n) == 2840, f'{name}: expected 2840 SSL images, got {n}'
    src = str(ck.get('source', ''))
    if src:
        assert '2840' in src.lower() or 'corrected' in src.lower(), f'{name}: unexpected provenance: {src}'
    print(f'{name}: clean checkpoint verified | SSL images={n} | source={src or "clean Colab 01"}')

print('\nPRE-FLIGHT AUDIT PASSED')

## 3. Deterministic labelled subsets

For 25% and 50%, the subset is stratified by proxy class. Each fraction × seed subset is generated once and reused unchanged across Random, ImageNet, MAE-style and EMA-distillation conditions.

In [ ]:
def stratified_fraction(records, fraction, seed):
    records = list(records)
    if fraction >= 1.0:
        return records
    labels = np.asarray([int(r['binary_label']) for r in records])
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=fraction, random_state=seed)
    idx, _ = next(splitter.split(np.zeros(len(records)), labels))
    return [records[i] for i in idx]

SUBSET_MANIFEST = os.path.join(Config.MANIFEST_DIR, 'experiment_A_label_subsets.json')
subset_map = {}
for frac in Config.LABEL_FRACTIONS:
    for seed in Config.DOWNSTREAM_SEEDS:
        recs = stratified_fraction(cubs_train_records, frac, seed)
        key = f'fraction_{int(frac*100)}_seed_{seed}'
        subset_map[key] = {
            'fraction': frac, 'seed': seed, 'n': len(recs),
            'class_0': sum(int(r['binary_label']) == 0 for r in recs),
            'class_1': sum(int(r['binary_label']) == 1 for r in recs),
            'image_paths': [r['image_path'] for r in recs]
        }

with open(SUBSET_MANIFEST, 'w') as f:
    json.dump(subset_map, f, indent=2)

print('Saved:', SUBSET_MANIFEST)
for k,v in subset_map.items():
    print(k, 'n=', v['n'], 'class0=', v['class_0'], 'class1=', v['class_1'])

## 4. Dataset, transforms and frozen encoder models

In [ ]:
MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
train_tf = T.Compose([
    T.Resize((224,224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(5),
    T.ToTensor(),
    T.Normalize(MEAN,STD)
])
eval_tf = T.Compose([
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(MEAN,STD)
])

class CUBSClassDataset(Dataset):
    def __init__(self, records, train=False):
        self.records=list(records); self.train=train
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r=self.records[i]
        x=Image.open(r['image_path']).convert('RGB')
        x=(train_tf if self.train else eval_tf)(x)
        y=torch.tensor(int(r['binary_label']), dtype=torch.long)
        return x,y

class FrozenEncoderClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__(); self.encoder=encoder; self.classifier=nn.Linear(encoder.num_features,2)
    def forward(self,x): return self.classifier(self.encoder(x))

def _load_clean_encoder_checkpoint(path):
    ck=torch.load(path,map_location='cpu')
    assert 'encoder_state_dict' in ck
    return ck['encoder_state_dict']

def build_model(kind):
    if kind == 'Random':
        encoder=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
    elif kind == 'ImageNet':
        encoder=timm.create_model(Config.ARCH, pretrained=True, num_classes=0)
    elif kind == 'MAE_clean_2840':
        encoder=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
        encoder.load_state_dict(_load_clean_encoder_checkpoint(MAE_CKPT), strict=True)
    elif kind == 'IJEPA_clean_2840':
        encoder=timm.create_model(Config.ARCH, pretrained=False, num_classes=0)
        encoder.load_state_dict(_load_clean_encoder_checkpoint(IJEPA_CKPT), strict=True)
    else:
        raise ValueError(kind)
    for p in encoder.parameters(): p.requires_grad=False
    return FrozenEncoderClassifier(encoder).to(DEVICE)

ENCODERS=['Random','ImageNet','MAE_clean_2840','IJEPA_clean_2840']
print('Encoder conditions:', ENCODERS)

## RAM image cache for faster resume

This does **not** change the Experiment A protocol. It only avoids repeatedly decoding the same CUBS image files from Google Drive. Random training augmentation remains applied at access time.

In [ ]:
# Cache all 2,176 decoded RGB CUBS images in RAM once.
ALL_CUBS_RECORDS = cubs_train_records + cubs_val_records + cubs_test_records
IMAGE_RAM_CACHE = {}
for i, r in enumerate(ALL_CUBS_RECORDS, 1):
    p = r['image_path']
    if p not in IMAGE_RAM_CACHE:
        with Image.open(p) as im:
            IMAGE_RAM_CACHE[p] = im.convert('RGB').copy()
    if i % 500 == 0:
        print(f'Cached {i}/{len(ALL_CUBS_RECORDS)} images')
print('RAM image cache ready:', len(IMAGE_RAM_CACHE), 'images')

def _cached_getitem(self, i):
    r = self.records[i]
    x = IMAGE_RAM_CACHE[r['image_path']].copy()
    x = (train_tf if self.train else eval_tf)(x)
    y = torch.tensor(int(r['binary_label']), dtype=torch.long)
    return x, y

CUBSClassDataset.__getitem__ = _cached_getitem
print('Dataset now reads decoded images from RAM; augmentation unchanged.')


## 5. Metrics and single-run training

Only the linear classifier head is optimized. Early stopping uses the fixed validation set; the test set is not used for model selection.

In [ ]:
@torch.no_grad()
def class_metrics(model, loader):
    model.eval(); ys=[]; probs=[]; preds=[]
    for x,y in loader:
        logits=model(x.to(DEVICE)); p=torch.softmax(logits,1)[:,1]; pred=logits.argmax(1)
        ys.extend(y.numpy()); probs.extend(p.cpu().numpy()); preds.extend(pred.cpu().numpy())
    ys=np.asarray(ys); probs=np.asarray(probs); preds=np.asarray(preds)
    auc=roc_auc_score(ys,probs) if len(np.unique(ys))==2 else np.nan
    return {
        'auc':float(auc), 'accuracy':float(accuracy_score(ys,preds)),
        'f1':float(f1_score(ys,preds,zero_division=0)),
        'precision':float(precision_score(ys,preds,zero_division=0)),
        'recall':float(recall_score(ys,preds,zero_division=0))
    }

def run_id(kind, fraction, seed):
    return f'{kind}_labels{int(round(fraction*100))}_seed{seed}_frozen'

def train_one(kind, fraction, seed):
    set_seed(seed)
    tr_records=stratified_fraction(cubs_train_records, fraction, seed)
    tr=DataLoader(CUBSClassDataset(tr_records,True), batch_size=Config.CLASS_BATCH_SIZE, shuffle=True, num_workers=0)
    va=DataLoader(CUBSClassDataset(cubs_val_records,False), batch_size=Config.CLASS_BATCH_SIZE, shuffle=False, num_workers=0)
    te=DataLoader(CUBSClassDataset(cubs_test_records,False), batch_size=Config.CLASS_BATCH_SIZE, shuffle=False, num_workers=0)

    rid=run_id(kind,fraction,seed)
    state_path=os.path.join(Config.STATE_DIR, f'class_{rid}.pth')
    best_path=os.path.join(Config.CHECKPOINT_DIR, f'class_{rid}_best.pth')
    history_path=os.path.join(Config.REVIEW_DIR, f'class_history_{rid}.csv')

    model=build_model(kind)
    opt=torch.optim.AdamW(model.classifier.parameters(), lr=Config.CLASS_LR, weight_decay=1e-4)
    start_epoch=0; best=-1.0; wait=0; best_sd=None; history=[]

    if os.path.exists(state_path):
        st=torch.load(state_path,map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['optimizer'])
        start_epoch=st['epoch']+1; best=st['best_score']; wait=st['wait']; best_sd=st['best_model']; history=st.get('history',[])
        print('RESUME',rid,'from epoch',start_epoch+1)

    for epoch in range(start_epoch, Config.CLASS_EPOCHS):
        model.train(); model.encoder.eval(); losses=[]
        for x,y in tr:
            x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
            loss=F.cross_entropy(model(x),y); loss.backward(); opt.step(); losses.append(loss.item())
        vm=class_metrics(model,va); score=vm['auc'] if not np.isnan(vm['auc']) else vm['accuracy']
        if score > best + 1e-5:
            best=score; best_sd=copy.deepcopy(model.state_dict()); wait=0
            torch.save({'model_state_dict':best_sd,'run_id':rid,'val_metrics':vm},best_path)
        else:
            wait += 1
        history.append({'epoch':epoch+1,'train_loss':float(np.mean(losses)),**{f'val_{k}':v for k,v in vm.items()}})
        pd.DataFrame(history).to_csv(history_path,index=False)
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':opt.state_dict(),'best_score':best,'best_model':best_sd,'wait':wait,'history':history},state_path)
        print(f'{rid}: epoch={epoch+1:02d} loss={np.mean(losses):.4f} val_auc={vm["auc"]:.4f} wait={wait}')
        if wait >= Config.CLASS_PATIENCE: break

    assert best_sd is not None
    model.load_state_dict(best_sd)
    tm=class_metrics(model,te)
    result={'encoder':kind,'label_fraction':fraction,'seed':seed,'n_train_labels':len(tr_records),'frozen_encoder':True,'best_val_auc':best,'run_id':rid,**tm}
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

## 6. Run the 36-condition Experiment A matrix

This cell is restart-safe. Completed rows in the result CSV are skipped. If Colab disconnects during a run, its saved resume state is used when the cell is run again.

## Resume check

Existing results are reused. Completed runs are skipped and only missing runs are launched.

In [ ]:
RESULT_CSV=os.path.join(Config.RESULT_DIR,'experiment_A_clean_frozen_label_efficiency.csv')
if os.path.exists(RESULT_CSV):
    _existing = pd.read_csv(RESULT_CSV)
    print(f'Existing completed runs found: {len(_existing)}/36')
    display(_existing[['encoder','label_fraction','seed','run_id']].sort_values(['label_fraction','seed','encoder']))
else:
    print('No existing result CSV found.')


In [ ]:
RESULT_CSV=os.path.join(Config.RESULT_DIR,'experiment_A_clean_frozen_label_efficiency.csv')

def load_results():
    return pd.read_csv(RESULT_CSV) if os.path.exists(RESULT_CSV) else pd.DataFrame()

def save_result(result):
    df=load_results()
    if len(df):
        same=(df['encoder'].eq(result['encoder']) &
              np.isclose(df['label_fraction'].astype(float),float(result['label_fraction'])) &
              df['seed'].astype(int).eq(int(result['seed'])))
        df=df.loc[~same].copy()
    df=pd.concat([df,pd.DataFrame([result])],ignore_index=True)
    df.to_csv(RESULT_CSV,index=False)
    return df

def completed(df,kind,frac,seed):
    if not len(df): return False
    return bool((df['encoder'].eq(kind) & np.isclose(df['label_fraction'].astype(float),frac) & df['seed'].astype(int).eq(seed)).any())

RUN_ALL=False  # Change to True only when ready.

if RUN_ALL:
    for frac in Config.LABEL_FRACTIONS:
        for seed in Config.DOWNSTREAM_SEEDS:
            # Same fraction+seed subset is used for every encoder below.
            for kind in ENCODERS:
                df=load_results()
                if completed(df,kind,frac,seed):
                    print('SKIP completed:',kind,frac,seed); continue
                print('\n'+'='*80)
                print('RUN:',kind,'fraction=',frac,'seed=',seed)
                r=train_one(kind,frac,seed)
                df=save_result(r)
                print('TEST:',r)
                print(f'Completed {len(df)}/36 runs | saved: {RESULT_CSV}')
else:
    print('Pre-flight mode only. Set RUN_ALL=True to launch the 36 runs.')
    df=load_results()
    print('Already completed:',len(df),'/36')

## 7. Results and mean ± SD summary

In [ ]:
df=load_results()
if not len(df):
    print('No completed Experiment A runs yet.')
else:
    df=df.sort_values(['label_fraction','encoder','seed'])
    display(df)
    print(f'Completed: {len(df)}/36')
    metrics_cols=['auc','accuracy','f1','precision','recall']
    summary=df.groupby(['encoder','label_fraction'])[metrics_cols].agg(['mean','std'])
    display(summary)
    summary_path=os.path.join(Config.REVIEW_DIR,'experiment_A_clean_summary_mean_sd.csv')
    summary.to_csv(summary_path)
    print('Saved summary:',summary_path)

## 8. Experiment A checks

Run after all 36 rows are complete. This checks the factorial design and confirms that the two SSL checkpoints are available.

In [ ]:
df=load_results()
assert len(df)==36, f'Expected 36 completed runs, found {len(df)}'
assert set(df['encoder'])==set(ENCODERS)
assert set(np.round(df['label_fraction'].astype(float),2))==set(Config.LABEL_FRACTIONS)
assert set(df['seed'].astype(int))==set(Config.DOWNSTREAM_SEEDS)
assert df.groupby(['encoder','label_fraction','seed']).size().eq(1).all()
assert df['frozen_encoder'].astype(bool).all()
assert os.path.exists(IJEPA_CKPT) and os.path.exists(MAE_CKPT)

print('='*65)
print('EXPERIMENT A CLEAN AUDIT PASSED')
print('4 encoders × 3 label fractions × 3 seeds = 36 runs')
print('Encoders frozen: YES')
print('Fractions: 25%, 50%, 100%')
print('Seeds: 42, 43, 44')
print('Clean MAE checkpoint: YES')
print('Clean I-JEPA checkpoint: YES')
print('Result:',RESULT_CSV)
print('='*65)